# Results explorer

Reads the [results store](../autobus-docs/docs/concepts/experiment.md) and
**nothing else** — no simulation runs here and no CARLA server is needed, so
it is safe to re-run while a campaign is still going on another machine. Each
re-run simply picks up whatever has synced since.

It differs from `results.ipynb` in one way: that notebook shows you the
scenario with the most runs, this one shows you **exactly the runs you ask
for**. Everything it reports is selected by the constants in *Filters* below,
and everything it draws is faceted by `GROUP_BY`, so the same notebook reads a
four-town CARLA campaign, a load sweep, or one policy on one map.

## What it reports

1. **Inventory** — what matched, and how many seeds each policy has.
2. **Provenance** — which commits produced the runs. Read this first: a run's
   identity does not include the code that produced it, so runs from before a
   behaviour change look current while being stale.
3. **KPI tables and bars** — the four headline KPIs per group, with 95% CIs.
4. **§3.2.2** — the fulfilment gate and the sum of percentage differences
   against the fixed-route baseline.
5. **Paired tests** — per-seed differences against the baseline, which is the
   comparison the campaign's shared seeds were run for.
6. **Breach rate** — fulfilled riders whose wait exceeded the promised window.
7. **Distributions** — every fulfilled passenger's wait and ride, pooled.
8. **Precision** — how far each KPI is from a 5% CI half-width, and how many
   seeds would close the gap.
9. **Driving health** and **policy work** — the backend and policy counters.
10. **Sweep** — any KPI against any swept parameter, on request.

## Two traps this notebook cannot fix for you

- **Never read wait or ride without fulfilment beside them.** Both are
  computed over *fulfilled* requests only, so a policy that abandons its hard
  riders posts better numbers. That is why the gate is in the §3.2.2 table.
- **Breach rate is structurally zero when the window equals patience**, since
  anyone who waits that long has already cancelled. The breach section says so
  when it detects it.

## House style

The project's validated categorical palette: three slots, all-pairs CVD
separation 9.2 and normal-vision 27.6 on the surface below. Three is the cap
that clears all pairs, so a fourth policy gets grey and its own row in the
table rather than an invented hue. One slot falls just under 3:1 contrast, and
the accepted relief is that **every chart ships beside its table**.

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler

# Validated palette slots 1-3, plus a neutral for any series beyond the third.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
OVERFLOW = "#8a8a86"
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT,
    "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "figure.dpi": 110,
})

## Filters

Everything below reports on the runs these select, and only those.

**Every filter takes `None` to mean "do not filter on this"**, a single value,
or a list of accepted values. They are combined with AND.

`EXTRA_FILTERS` is the escape hatch for anything not named here: keys are
dotted paths into a stored run, exactly as the store writes them —
`identity.world.<key>`, `identity.trial.<key>`,
`identity.policy_params.<key>`, `provenance.<key>`. The *Inventory* section
prints every path actually present when nothing matches, so a typo is
visible rather than silent.

In [ ]:
from src.experiment.store import DEFAULT_RUNS_DIR

RUNS_DIR = DEFAULT_RUNS_DIR

# --- the world -------------------------------------------------------
# "MockBackend" for the synthetic map, "CarlaBackend" for a town.
BACKEND = None
TOWNS = None            # CARLA only, e.g. ["Town01", "Town03"]
MAPS = None             # synthetic only, e.g. "planar"
MAP_SEEDS = None        # synthetic only
STOP_DENSITY = None     # synthetic only; the campus map needs 0.5
NPC_VEHICLES = None     # CARLA only
STEERING = None         # CARLA only, e.g. "direct-drive-v1"

# --- the scenario ----------------------------------------------------
POLICIES = None         # labels, e.g. ["Fixed route", "Reordering W750"]
POLICY_CLASSES = None   # class names, when labels vary between campaigns
MAX_REQUESTS = None
DEMAND_INTERVAL_S = None
# Passenger patience: run_trial's max_wait_s, when a rider gives up. NOT the
# policy's service window, which is POLICY_WINDOW below.
PATIENCE = None
CAPACITY = None
COMMITMENT = None
N_STOPS = None
SEEDS = None            # e.g. range(1, 11); a list of demand seeds

# --- when it was run -------------------------------------------------
GIT_COMMITS = None      # e.g. "20ba830"; short hashes as recorded
# Drop anything finished before this instant. An ISO-8601 prefix is enough:
# "2026-08-16" keeps that day onwards. The store's timestamps are UTC.
FINISHED_SINCE = None

# Anything else, by dotted path.
EXTRA_FILTERS: dict[str, object] = {}

# --- how to report ---------------------------------------------------
# Dotted path every table and chart is faceted by. "auto" picks the town when
# the runs have one and the map otherwise; None pools everything into one
# block. Point it at "identity.trial.max_requests" to read a load sweep.
GROUP_BY = "auto"

# The comparator for the design review's criteria. None disables those
# sections.
BASELINE_POLICY = "Fixed route"

# The service window the policies promised, for the breach rate. None reads
# it from each policy's own max_wait_s parameter, which is right whenever the
# window is what varies between policies.
POLICY_WINDOW = None

# Keep only the seeds every policy has. True is right for any comparison
# between policies -- an interrupted campaign otherwise leaves policies with
# different seed sets, and comparing those is wrong with no error. Set False
# for a sweep, or when looking at one policy.
PAIRED = True

CONFIDENCE = 0.95
FULFILMENT_GATE = 0.80      # the design review's >= 80%
PRECISION_TARGET = 0.05     # CI half-width as a fraction of the mean

# A dotted path to plot KPIs against, for the sweep section at the end.
# None skips it.
SWEEP_X = None

## Load

One pass over the store, filtering as it reads rather than after, since the
store holds thousands of runs and only the matching ones are worth
rebuilding.

In [ ]:
import json
import math
import statistics
from collections import defaultdict
from collections.abc import Mapping, Sequence

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

from src.experiment.store import RunRecord, to_results
from src.experiment.summary import summarize_results

from IPython.display import Markdown, display


def dotted(data: Mapping[str, object], path: str) -> object:
    """
    Read a nested value out of a stored run by dotted path.
    :param data: One decoded run file.
    :param path: A path such as ``identity.trial.max_requests``.
    :return: The value, or None if any step is missing.
    """
    current: object = data
    for part in path.split("."):
        if not isinstance(current, Mapping) or part not in current:
            return None
        current = current[part]
    return current


def accepted(value: object, wanted: object) -> bool:
    """
    Test one value against one filter setting.

    A list, tuple, set or range means "any of these"; anything else means
    equality. Numbers are compared as floats so that 900 and 900.0 are the
    same setting, which is how the store hashes them too.
    :param value: What the run recorded.
    :param wanted: What the filter asked for.
    :return: True when the run passes this filter.
    """
    if isinstance(wanted, (list, tuple, set, range)):
        return any(accepted(value, item) for item in wanted)
    if isinstance(value, bool) or isinstance(wanted, bool):
        return value == wanted
    if isinstance(value, (int, float)) and isinstance(wanted, (int, float)):
        return float(value) == float(wanted)
    return value == wanted


# Named constants mapped onto the paths they filter. Only the ones that were
# set take part.
FILTERS: dict[str, object] = {
    path: value
    for path, value in {
        "identity.world.backend": BACKEND,
        "identity.world.town": TOWNS,
        "identity.world.map": MAPS,
        "identity.world.map_seed": MAP_SEEDS,
        "identity.world.stop_density": STOP_DENSITY,
        "identity.world.npc_vehicles": NPC_VEHICLES,
        "identity.world.steering": STEERING,
        "identity.policy": POLICIES,
        "identity.policy_class": POLICY_CLASSES,
        "identity.trial.max_requests": MAX_REQUESTS,
        "identity.trial.demand_interval_s": DEMAND_INTERVAL_S,
        "identity.trial.max_wait_s": PATIENCE,
        "identity.trial.capacity": CAPACITY,
        "identity.trial.commitment": COMMITMENT,
        "identity.trial.n_stops": N_STOPS,
        "identity.seed": SEEDS,
        "provenance.git_commit": GIT_COMMITS,
        **EXTRA_FILTERS,
    }.items()
    if value is not None
}


def matches(data: Mapping[str, object]) -> bool:
    """
    Decide whether one stored run passes every filter.
    :param data: One decoded run file.
    :return: True when it should be reported on.
    """
    for path, wanted in FILTERS.items():
        if not accepted(dotted(data, path), wanted):
            return False
    if FINISHED_SINCE is not None:
        finished = dotted(data, "provenance.finished_at")
        if finished is None or str(finished) < str(FINISHED_SINCE):
            return False
    return True


def load_matching(directory) -> tuple[list[RunRecord], dict[str, dict]]:
    """
    Read the store once, keeping the runs the filters accept.

    Returns the raw dictionaries alongside the records because the filters,
    the grouping and the sweep all address runs by dotted path, and rebuilding
    a record into a dictionary for every lookup is wasted work.
    :param directory: Where runs are filed.
    :return: The matching records, and their raw form keyed by run id.
    """
    paths = sorted(directory.glob("*.json"))
    records: list[RunRecord] = []
    raw: dict[str, dict] = {}
    seen = 0
    bar = tqdm(paths, desc="reading store", unit="run")
    for path in bar:
        data = json.loads(path.read_text(encoding="utf-8"))
        seen += 1
        if not matches(data):
            continue
        record = RunRecord.from_dict(data)
        records.append(record)
        raw[record.run_id] = data
        bar.set_postfix(kept=len(records), scanned=seen)
    bar.close()
    return records, raw


records, RAW = load_matching(RUNS_DIR)
print(f"{len(records)} runs matched "
      f"{len(FILTERS) + (FINISHED_SINCE is not None)} filter(s)")

## Inventory

What matched, and whether it is enough to compare anything. When nothing
matched, every filterable path in the store is listed with its recorded
values, so a filter that is merely misspelled or set to a value nobody ever
ran is visible immediately.

In [ ]:
def available_paths(directory, limit: int = 400) -> pd.DataFrame:
    """
    Survey which paths and values exist in the store.

    Reads a sample rather than everything: this runs when a filter matched
    nothing, and the point is to show what could have been asked for.
    :param directory: Where runs are filed.
    :param limit: How many run files to sample.
    :return: One row per path, with its distinct values (truncated).
    """
    values: dict[str, set] = defaultdict(set)
    paths = sorted(directory.glob("*.json"))[:limit]
    for path in tqdm(paths, desc="sampling store", unit="run"):
        data = json.loads(path.read_text(encoding="utf-8"))
        for section in ("identity", "provenance"):
            for key, value in (data.get(section) or {}).items():
                if isinstance(value, Mapping):
                    for sub, item in value.items():
                        values[f"{section}.{key}.{sub}"].add(str(item))
                else:
                    values[f"{section}.{key}"].add(str(value))
    rows = [
        {"path": path,
         "distinct": len(items),
         "values": ", ".join(sorted(items)[:8])
                   + (" ..." if len(items) > 8 else "")}
        for path, items in sorted(values.items())
    ]
    return pd.DataFrame(rows).set_index("path")


HAVE_RUNS = bool(records)

if not HAVE_RUNS:
    display(Markdown(
        "**Nothing matched.** Either the campaign has not synced yet, or a "
        "filter names a value nobody ran. What the store actually holds "
        "(sampled):"))
    display(available_paths(RUNS_DIR))
else:
    frame = pd.json_normalize([RAW[r.run_id] for r in records])
    display(frame.groupby(
        ["identity.policy"], dropna=False).agg(
            runs=("run_id", "count"),
            seeds=("identity.seed", "nunique"),
        ).sort_values("runs", ascending=False))

### Grouping

`GROUP_BY` decides how every section below is faceted. Groups with fewer than
two policies still get their tables — a single policy is a perfectly good
thing to look at — but the comparison sections skip them and say so.

In [ ]:
def resolve_group_path(records: Sequence[RunRecord]) -> str | None:
    """
    Settle which path the report is faceted by.

    ``"auto"`` prefers the town, because a CARLA campaign's whole question is
    per-town, and falls back to the synthetic map name.
    :param records: The matching runs.
    :return: A dotted path, or None to pool everything.
    """
    if GROUP_BY != "auto":
        return GROUP_BY
    for candidate in ("identity.world.town", "identity.world.map"):
        if any(dotted(RAW[r.run_id], candidate) is not None
               for r in records):
            return candidate
    return None


def group_runs(
    records: Sequence[RunRecord],
) -> dict[str, list[RunRecord]]:
    """
    Split the matching runs into the blocks the report is written in.
    :param records: The matching runs.
    :return: Group label to its runs, in sorted label order.
    """
    if GROUP_PATH is None:
        return {"all runs": list(records)}
    grouped: dict[str, list[RunRecord]] = defaultdict(list)
    for record in records:
        grouped[str(dotted(RAW[record.run_id], GROUP_PATH))].append(record)
    return dict(sorted(grouped.items()))


GROUP_PATH = resolve_group_path(records) if HAVE_RUNS else None
GROUPS = group_runs(records) if HAVE_RUNS else {}

if HAVE_RUNS:
    rows = []
    for label, runs in GROUPS.items():
        policies = {r.identity.policy for r in runs}
        seeds = {r.identity.seed for r in runs}
        shared = set.intersection(*(
            {r.identity.seed for r in runs if r.identity.policy == policy}
            for policy in policies
        )) if policies else set()
        # More than one run per (policy, seed) means the filters left two
        # different scenarios pooled together -- different maps, loads or
        # patience. Every paired statistic below then compares runs that were
        # never meant to be compared, so this is a warning and not a note.
        pairs = [(r.identity.policy, r.identity.seed) for r in runs]
        rows.append({
            "group": label,
            "runs": len(runs),
            "policies": len(policies),
            "seeds": len(seeds),
            "seeds all policies share": len(shared),
            "runs per policy+seed": round(len(pairs) / len(set(pairs)), 2),
        })
    inventory = pd.DataFrame(rows).set_index("group")
    display(inventory)
    print(f"faceted by {GROUP_PATH or 'nothing (pooled)'}"
          + ("; paired: only shared seeds are reported" if PAIRED
             else "; unpaired: every run counts"))

    crowded = inventory[inventory["runs per policy+seed"] > 1.0]
    if not crowded.empty:
        display(Markdown(
            "⚠ **More than one run per policy and seed** in "
            f"{', '.join(crowded.index)}. The filters have left several "
            "different scenarios pooled together, and every paired figure "
            "below then compares runs that share nothing but a seed number. "
            "Narrow the filters — the *Inventory* paths above list what "
            "varies — or set `GROUP_BY` to whatever distinguishes them."))

## Provenance

**A run's identity is what it was asked to do, deliberately not the code that
ran it** — so a run made before a behaviour change looks current while being
stale. Two commits in one group is not automatically wrong (a docs-only
commit changes nothing), but it is the first thing to check when a result
looks odd.

`PYTHONHASHSEED` matters because several policies break ties by set-iteration
order: a run recorded without it pinned cannot be reproduced exactly.

In [ ]:
if HAVE_RUNS:
    prov = frame[["provenance.git_commit",
                  "provenance.python_hash_seed"]].fillna("unset")
    display(prov.value_counts().to_frame("runs"))

    commits = set(prov["provenance.git_commit"])
    if len(commits) > 1:
        display(Markdown(
            f"⚠ **{len(commits)} different commits** produced these runs "
            f"({', '.join(sorted(commits))}). If any of them changed "
            "simulation behaviour, the older runs are stale and should be "
            "re-run with `overwrite=True` rather than compared."))

    unpinned = int((frame["provenance.python_hash_seed"].isna()).sum())
    if unpinned:
        print(f"{unpinned} runs were recorded without PYTHONHASHSEED "
              "pinned; their exact reproduction is not guaranteed")
    else:
        print("every matching run was recorded with PYTHONHASHSEED pinned")

    wall = frame["provenance.wall_seconds"].dropna()
    if not wall.empty:
        print(f"wall time: {wall.sum() / 3600:.1f} h total, "
              f"{wall.mean() / 60:.1f} min per trial on average")

## The KPIs

One table per group: every measured property, its mean across seeds, and its
95% confidence half-width as a percentage of that mean.

**Read fulfilment first.** Wait and ride are means over fulfilled requests
only, so a policy that gives up on its difficult riders looks good on both.

In [ ]:
def results_for(runs: Sequence[RunRecord]) -> dict:
    """
    Rebuild one group's runs into the shape the summary and plots take.
    :param runs: The runs in one group.
    :return: Policy label to its trial results.
    """
    return to_results(runs, paired=PAIRED)


RESULTS = {label: results_for(runs) for label, runs in GROUPS.items()}

for label, results in RESULTS.items():
    display(Markdown(f"### {label}"))
    if not results:
        display(Markdown("_No runs._"))
        continue
    display(Markdown(summarize_results(
        results, confidence=CONFIDENCE).markdown))

The four headline KPIs as bars, with 95% confidence intervals across seeds.
One panel per KPI so there is a single scale on every axis; a shorter bar is
better for wait and ride, a taller one for fulfilment and throughput.

In [ ]:
KPIS = [
    ("served_fraction", "Served fraction", 100.0, "%", "higher"),
    ("mean_wait_time", "Mean wait", 1.0, "s", "lower"),
    ("mean_ride_time", "Mean ride", 1.0, "s", "lower"),
    ("throughput", "Throughput", 1.0, "req/min", "higher"),
]


def kpi_values(runs, field: str, scale: float) -> list[float]:
    """
    Pull one KPI out of a policy's runs, dropping the runs that lack it.
    :param runs: One policy's trial results.
    :param field: A KPISnapshot attribute name.
    :param scale: Multiplier applied before reporting.
    :return: The recorded values.
    """
    return [getattr(r.kpi, field) * scale for r in runs
            if getattr(r.kpi, field) is not None]


def mean_and_half(values: Sequence[float]) -> tuple[float, float]:
    """
    Mean and confidence half-width of a sample.
    :param values: The per-seed values.
    :return: (mean, half-width); half-width is 0 for a single value.
    """
    mean = statistics.fmean(values)
    if len(values) < 2:
        return mean, 0.0
    critical = stats.t.ppf((1.0 + CONFIDENCE) / 2.0, len(values) - 1)
    return mean, critical * statistics.stdev(values) / math.sqrt(len(values))


def colour_for(index: int) -> str:
    """
    The palette slot for the nth policy.
    :param index: Position in the legend order.
    :return: A hex colour; anything past the third slot is neutral grey.
    """
    return SERIES[index] if index < len(SERIES) else OVERFLOW


def plot_kpis(results: dict, title: str):
    """
    Draw the four headline KPIs for one group as grouped bars.
    :param results: Policy label to its trial results.
    :param title: Figure title, normally the group label.
    :return: The figure, or None when there is nothing to draw.
    """
    labels = list(results)
    if not labels:
        return None
    figure, axes = plt.subplots(1, len(KPIS), figsize=(4.0 * len(KPIS), 3.4))
    width = 0.8 / len(labels)
    for axis, (field, name, scale, unit, better) in zip(axes, KPIS):
        for index, label in enumerate(labels):
            values = kpi_values(results[label], field, scale)
            if not values:
                continue
            mean, half = mean_and_half(values)
            axis.bar(index * width, mean, width * 0.92,
                     yerr=half or None, capsize=3,
                     color=colour_for(index), label=label)
        axis.set_title(f"{name} ({unit})")
        axis.set_xlabel(f"{better} is better")
        axis.set_xticks([])
        axis.grid(axis="y", alpha=0.6)
        axis.set_axisbelow(True)
    handles, names = axes[0].get_legend_handles_labels()
    figure.legend(handles, names, loc="lower center",
                  ncol=min(len(labels), 4), bbox_to_anchor=(0.5, -0.06))
    figure.suptitle(title, y=1.02)
    figure.tight_layout()
    return figure


for label, results in RESULTS.items():
    if plot_kpis(results, label) is not None:
        plt.show()

## §3.2.2 — the design review's criteria

Two things are asked for: fulfilment at or above **80%**, and a **positive sum
of percentage differences** across wait, ride and throughput against a
fixed-route bus.

The gate does the real work. The sum must never be read on its own, because
two of its three terms reward abandoning passengers.

In [ ]:
def percentage_differences(results: dict, baseline: str) -> pd.DataFrame:
    """
    Score every policy against the baseline on the review's three KPIs.

    Wait and ride count as improvements when they fall, throughput when it
    rises, so all three differences are signed the same way and can be summed.
    :param results: Policy label to its trial results.
    :param baseline: Label of the comparator.
    :return: One row per policy; empty when the baseline is absent.
    """
    if baseline not in results:
        return pd.DataFrame()

    def mean_of(runs, field: str) -> float:
        """Average one KPI over a policy's runs."""
        values = kpi_values(runs, field, 1.0)
        return statistics.fmean(values) if values else float("nan")

    base = results[baseline]
    rows = []
    for label, runs in results.items():
        fulfilment = mean_of(runs, "served_fraction")
        deltas = {}
        for field, name, sign in (("mean_wait_time", "wait", -1),
                                  ("mean_ride_time", "ride", -1),
                                  ("throughput", "throughput", 1)):
            reference = mean_of(base, field)
            value = mean_of(runs, field)
            deltas[name] = (
                float("nan") if not reference
                else sign * (value - reference) / reference * 100.0
            )
        total = sum(deltas.values())
        passes = fulfilment >= FULFILMENT_GATE and total > 0
        rows.append({
            "policy": label,
            "fulfilment %": fulfilment * 100.0,
            "wait Δ%": deltas["wait"],
            "ride Δ%": deltas["ride"],
            "throughput Δ%": deltas["throughput"],
            "sum Δ%": total,
            "gate": "pass" if fulfilment >= FULFILMENT_GATE else "FAIL",
            "§3.2.2": "pass" if passes else "fail",
        })
    return pd.DataFrame(rows).set_index("policy").round(1)


if BASELINE_POLICY is None:
    display(Markdown("_`BASELINE_POLICY` is None, so the criteria are not "
                     "scored._"))
else:
    for label, results in RESULTS.items():
        display(Markdown(f"### {label}"))
        table = percentage_differences(results, BASELINE_POLICY)
        if table.empty:
            display(Markdown(
                f"_No `{BASELINE_POLICY}` runs here to compare against._"))
        else:
            display(table)

## Paired differences against the baseline

The table above compares two averages. This compares the **same demand seed**
under two policies and averages the differences, which is what running every
policy on a shared seed set was for: it removes the seed-to-seed variation
that otherwise swamps the effect.

`p` is a two-sided paired t-test. With ten seeds it is under-powered for small
effects — a large p here means "not demonstrated at this sample size", not
"no difference".

In [ ]:
def paired_differences(runs: Sequence[RunRecord], baseline: str):
    """
    Compare each policy with the baseline seed by seed.
    :param runs: One group's runs.
    :param baseline: Label of the comparator.
    :return: One row per policy and KPI; empty when pairing is impossible.
    """
    by_policy: dict[str, dict[int, RunRecord]] = defaultdict(dict)
    for record in runs:
        by_policy[record.identity.policy][record.identity.seed] = record
    if baseline not in by_policy:
        return pd.DataFrame()

    base = by_policy[baseline]
    rows = []
    for label, seeded in by_policy.items():
        if label == baseline:
            continue
        shared = sorted(set(base) & set(seeded),
                        key=lambda s: (s is None, s))
        for field, name, scale, unit, better in KPIS:
            pairs = [
                (getattr(base[seed].kpi, field),
                 getattr(seeded[seed].kpi, field))
                for seed in shared
            ]
            pairs = [(a * scale, b * scale) for a, b in pairs
                     if a is not None and b is not None]
            if len(pairs) < 2:
                continue
            deltas = [b - a for a, b in pairs]
            mean, half = mean_and_half(deltas)
            p = stats.ttest_rel([b for _, b in pairs],
                                [a for a, _ in pairs]).pvalue
            improved = (mean < 0) if better == "lower" else (mean > 0)
            rows.append({
                "policy": label,
                "KPI": f"{name} ({unit})",
                "seeds": len(pairs),
                "mean Δ": mean,
                "95% CI": f"±{half:.3g}",
                "p": p,
                "direction": "better" if improved else "worse",
            })
    return pd.DataFrame(rows).set_index(["policy", "KPI"])


if BASELINE_POLICY is None:
    display(Markdown("_`BASELINE_POLICY` is None, so nothing is paired._"))
else:
    for label, runs in GROUPS.items():
        display(Markdown(f"### {label}"))
        table = paired_differences(runs, BASELINE_POLICY)
        if table.empty:
            display(Markdown(
                "_Not enough shared seeds to pair against "
                f"`{BASELINE_POLICY}` here._"))
        else:
            display(table.round({"mean Δ": 2, "p": 4}))

## Breach rate

A breach is a **fulfilled** rider whose wait exceeded the window the policy
promised. It is the honesty check on the promise.

It is structurally zero whenever the window equals passenger patience, since
anyone who waited that long has already cancelled and is not fulfilled. The
cell says so when it detects that, rather than letting a column of zeroes
read as a result.

In [ ]:
def window_for(record: RunRecord) -> float | None:
    """
    The service window a run promised.

    Prefers the policy's own ``max_wait_s`` parameter, since that is the
    promise it planned against and it is what varies across a window sweep.
    :param record: One run.
    :return: The window in seconds, or None when the policy declares none.
    """
    if POLICY_WINDOW is not None:
        return float(POLICY_WINDOW)
    declared = record.identity.policy_params.get("max_wait_s")
    return None if declared is None else float(declared)


def breach_table(runs: Sequence[RunRecord]) -> pd.DataFrame:
    """
    Breach rate per policy, over fulfilled riders.
    :param runs: One group's runs.
    :return: One row per policy.
    """
    rows = []
    for record in runs:
        window = window_for(record)
        if window is None or not record.wait_times:
            continue
        breaches = sum(w > window for w in record.wait_times)
        rows.append({
            "policy": record.identity.policy,
            "window s": window,
            "patience s": record.identity.trial.get("max_wait_s"),
            "breach %": 100.0 * breaches / len(record.wait_times),
            "worst wait s": max(record.wait_times),
        })
    if not rows:
        return pd.DataFrame()
    table = pd.DataFrame(rows)
    return table.groupby(["policy", "window s", "patience s"],
                         dropna=False).mean().round(2)


for label, runs in GROUPS.items():
    display(Markdown(f"### {label}"))
    table = breach_table(runs)
    if table.empty:
        display(Markdown(
            "_No policy here declares a service window, and "
            "`POLICY_WINDOW` is None, so there is nothing to breach._"))
        continue
    display(table)
    blind = [
        index for index in table.index
        if index[2] is not None and float(index[1]) >= float(index[2])
    ]
    if blind:
        display(Markdown(
            "⚠ The window is not tighter than patience for "
            f"{', '.join(str(i[0]) for i in blind)}, so **their breach rate "
            "is structurally zero** and measures nothing. Tighten the window "
            "or raise patience to measure it."))

## Where the time actually goes

Averages hide the tail. These are every fulfilled passenger's wait and ride
pooled across the group's seeds: the box spans the quartiles, the whiskers the
5th–95th percentile, the notch the median.

In [ ]:
def plot_spread(runs: Sequence[RunRecord], attribute: str, title: str):
    """
    Box plot of a per-passenger distribution, one row per policy.
    :param runs: One group's runs.
    :param attribute: ``wait_times`` or ``ride_times``.
    :param title: Chart title.
    :return: The figure, or None when nothing was recorded.
    """
    pooled: dict[str, list[float]] = defaultdict(list)
    for record in runs:
        pooled[record.identity.policy].extend(getattr(record, attribute))
    pooled = {k: v for k, v in pooled.items() if v}
    if not pooled:
        return None
    order = sorted(pooled, key=lambda k: statistics.median(pooled[k]))
    figure, axis = plt.subplots(figsize=(9, 0.5 * len(order) + 1.4))
    boxes = axis.boxplot([pooled[k] for k in order],
                         orientation="horizontal", notch=True,
                         whis=(5, 95), showfliers=False, widths=0.6,
                         patch_artist=True)
    for index, box in enumerate(boxes["boxes"]):
        box.set_facecolor(colour_for(index))
        box.set_alpha(0.75)
        box.set_edgecolor(INK_SOFT)
    axis.set_yticklabels(order)
    axis.set_xlabel("seconds")
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.6)
    axis.set_axisbelow(True)
    figure.tight_layout()
    return figure


def spread_table(runs: Sequence[RunRecord], attribute: str) -> pd.DataFrame:
    """
    The percentiles behind the box plot, since the chart needs its table.
    :param runs: One group's runs.
    :param attribute: ``wait_times`` or ``ride_times``.
    :return: One row per policy.
    """
    pooled: dict[str, list[float]] = defaultdict(list)
    for record in runs:
        pooled[record.identity.policy].extend(getattr(record, attribute))
    rows = []
    for label, values in sorted(pooled.items()):
        if not values:
            continue
        quantiles = np.percentile(values, [5, 25, 50, 75, 95])
        rows.append({
            "policy": label,
            "n": len(values),
            "p5": quantiles[0], "p25": quantiles[1], "median": quantiles[2],
            "p75": quantiles[3], "p95": quantiles[4],
            "max": max(values),
        })
    return pd.DataFrame(rows).set_index("policy").round(1)


for label, runs in GROUPS.items():
    display(Markdown(f"### {label}"))
    for attribute, title in (("wait_times", "Wait time"),
                             ("ride_times", "Ride time")):
        if plot_spread(runs, attribute, f"{title} — {label}") is not None:
            plt.show()
            display(spread_table(runs, attribute))

## Is the sample big enough?

The project's bar is a confidence half-width below **5% of the mean** on every
KPI. Where it is missed, the last column says how many seeds would reach it,
on the sample's own standard deviation — and extending a campaign only runs
the seeds that are missing, since everything recorded is skipped.

In [ ]:
def precision_table(results: dict) -> pd.DataFrame:
    """
    Achieved confidence half-width per policy and KPI, against the target.
    :param results: Policy label to its trial results.
    :return: One row per policy and KPI.
    """
    rows = []
    for label, runs in sorted(results.items()):
        for field, name, scale, unit, _ in KPIS:
            values = kpi_values(runs, field, scale)
            if len(values) < 2:
                continue
            mean, half = mean_and_half(values)
            if mean == 0:
                continue
            fraction = abs(half / mean)
            # n scales as 1/sqrt, so the seeds needed for the target follow
            # from the sample's own spread. A rough guide, not a promise:
            # it reuses this sample's sigma.
            needed = math.ceil(len(values) * (fraction
                                              / PRECISION_TARGET) ** 2)
            rows.append({
                "policy": label,
                "KPI": f"{name} ({unit})",
                "runs": len(values),
                "mean": mean,
                "CI half": half,
                "% of mean": fraction * 100.0,
                "meets target": "yes" if fraction <= PRECISION_TARGET
                                else "no",
                "seeds for target": len(values) if fraction
                                    <= PRECISION_TARGET else needed,
            })
    return pd.DataFrame(rows).set_index(["policy", "KPI"])


for label, results in RESULTS.items():
    display(Markdown(f"### {label}"))
    table = precision_table(results)
    if table.empty:
        display(Markdown("_Fewer than two seeds; no interval to report._"))
    else:
        display(table.round(2))

## Was the driving healthy?

Only CARLA records these. **`repaths` is the one to watch**: under direct
drive the red-light and queueing exemptions cover every legitimate stop, so a
repath means the bus physically stopped making progress. `unroutable_legs`
means the planner could not find a path at all, which invalidates the leg
rather than merely slowing it.

In [ ]:
DIAGNOSTICS = ["arrivals", "legs_started", "distance_m", "repaths",
               "replans", "route_drops", "unroutable_legs", "queued_ticks",
               "red_light_ticks", "ticks", "walker_spawn_failures",
               "stops_without_shoulder"]


def health_table(runs: Sequence[RunRecord]) -> pd.DataFrame:
    """
    Average the backend's counters per policy.
    :param runs: One group's runs.
    :return: One row per policy; empty when no run reported any.
    """
    rows = []
    for record in runs:
        if not record.backend_stats:
            continue
        row = {"policy": record.identity.policy}
        row.update({key: record.backend_stats.get(key)
                    for key in DIAGNOSTICS
                    if key in record.backend_stats})
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).groupby("policy").mean().round(1)


any_health = False
for label, runs in GROUPS.items():
    table = health_table(runs)
    if table.empty:
        continue
    any_health = True
    display(Markdown(f"### {label}"))
    display(table)
    stuck = sum(r.backend_stats.get("repaths", 0) for r in runs)
    lost = sum(r.backend_stats.get("unroutable_legs", 0) for r in runs)
    print(f"{stuck:.0f} repaths and {lost:.0f} unroutable legs across "
          f"{len(runs)} runs"
          + ("  (clean)" if stuck == 0 and lost == 0 else "  <-- look"))

if not any_health:
    display(Markdown("_No backend statistics recorded — these runs were "
                     "not driven in CARLA._"))

## What the policy did

The insertion policies report their own work: how many candidate positions
they tried, how many were accepted, and how much cost the repair passes
recovered. Useful for telling "the policy had no room" apart from "the policy
had room and did not use it".

In [ ]:
def policy_work(runs: Sequence[RunRecord]) -> pd.DataFrame:
    """
    Average the policy's own counters per policy label.
    :param runs: One group's runs.
    :return: One row per policy; empty when no run reported any.
    """
    rows = [
        {"policy": record.identity.policy, **record.policy_stats}
        for record in runs if record.policy_stats
    ]
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).groupby("policy").mean().round(2)


any_work = False
for label, runs in GROUPS.items():
    table = policy_work(runs)
    if table.empty:
        continue
    any_work = True
    display(Markdown(f"### {label}"))
    display(table)

if not any_work:
    display(Markdown("_None of these policies reported per-run statistics._"))

## Sweep

Set `SWEEP_X` to any dotted path that varies across the matched runs — for
instance `identity.trial.max_requests` for a load sweep, or
`identity.policy_params.max_wait_s` for a window sweep — and every KPI is
plotted against it, with a confidence band across the seeds at each point.

**Filter everything else down first.** The band is computed over whatever is
left, so a parameter nobody meant to vary is silently averaged into the
spread and the chart still looks perfectly reasonable.

In [ ]:
if SWEEP_X is None:
    display(Markdown("_`SWEEP_X` is None, so no sweep is plotted._"))
elif not HAVE_RUNS:
    display(Markdown("_Nothing matched, so there is nothing to sweep._"))
elif SWEEP_X not in frame.columns:
    display(Markdown(f"_`{SWEEP_X}` is not a column of these runs._"))
elif frame[SWEEP_X].nunique() < 2:
    display(Markdown(
        f"_Only one value of `{SWEEP_X}` matched, so there is no sweep. "
        "Widen the filters._"))
else:
    from src.experiment import plot_parameter_sweep
    for field, name, scale, unit, _ in KPIS:
        plot_parameter_sweep(frame, x=SWEEP_X, y=f"kpi.{field}",
                             hue="identity.policy",
                             title=f"{name} against {SWEEP_X.split('.')[-1]}",
                             confidence=int(CONFIDENCE * 100))
        plt.show()